# HW05 Demo

Reconstructs lensless-camera measurements with our best model modular-prepost, but you can choose other models too

`DATA_URL` is a Google-Drive `.zip` containing `lensless/<id>.png`, `masks/<id>.npy`, and optionally `lensed/<id>.png` (ground truth). The notebook clones the repo, installs deps, downloads the checkpoint, reconstructs your samples via `inference.py`, visualizes them, and (if `lensed/` exists) prints PSNR/SSIM/LPIPS/MSE via `calculate_metrics.py`.

In [ ]:
REPO_URL = "https://github.com/ndrew1337/lensless_hw.git"
CKPT_REPO = "PUFL/hw05-lensless"
CKPT_FILE = "modular-prepost.pth" # checkpoint filename in repo
DATA_URL = "https://drive.google.com/file/d/15EEa4-3pV7hmeDw5GWDH36Nh3so-Ahhu/view?usp=sharing" # GDrive .zip link (replace with your own)
MODEL = "modular_prepost" # model config to run

## 1. Clone the repository & install dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path

WORK = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORK / "hw05_lensless"
if not (REPO_DIR / "inference.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Download the trained checkpoint

In [ ]:
from huggingface_hub import hf_hub_download

CKPT = hf_hub_download(repo_id=CKPT_REPO, filename=CKPT_FILE)
print(f"Checkpoint ({Path(CKPT).stat().st_size/1e6:.0f} MB): {CKPT}")

## 3. Download & unzip dataset 
The zip must contain `lensless/` and `masks/` (and optionally `lensed/` ground truth).

In [ ]:
import gdown, zipfile, re

DATA_DIR = Path("data/custom")
DATA_DIR.mkdir(parents=True, exist_ok=True)
m = re.search(r"/d/([\w-]+)", DATA_URL) or re.search(r"id=([\w-]+)", DATA_URL)
file_id = m.group(1) if m else DATA_URL
gdown.download(id=file_id, output="data.zip", quiet=False)
with zipfile.ZipFile("data.zip") as z:
    z.extractall(DATA_DIR)
# if the archive wrapped everything in one top-level folder, go down
sub = [p for p in DATA_DIR.iterdir() if p.is_dir() and (p / "lensless").exists()]
if sub:
    DATA_DIR = sub[0]
n = len(list((DATA_DIR / "lensless").glob("*.png")))
print("Data dir:", DATA_DIR)
print("lensless samples:", n)

## 4. Reconstruct via `inference.py`
Runs the model over every measurement and writes reconstructions to `reconstructions/test/`.

In [ ]:
RECON = str(REPO_DIR / "reconstructions")
cmd = (
    f"python inference.py --config-name=inference_admm100 model={MODEL} "
    f"datasets=custom_dir datasets.test.data_dir={DATA_DIR} dataloader.batch_size=1 "
    f"inferencer.skip_model_load=false inferencer.from_pretrained={CKPT} "
    f"inferencer.save_path={RECON}"
)
os.system(cmd)
print("Reconstructions at: ", Path(RECON, "test").resolve())

## 5. Visualize — lensless measurement vs reconstruction (vs ground truth if available)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

recon_dir = Path(RECON, "test")
has_gt = (DATA_DIR / "lensed").exists()
ids = sorted(p.stem for p in recon_dir.glob('*.png'))[:4]
ncol = 3 if has_gt else 2
fig, axes = plt.subplots(len(ids), ncol, figsize=(4 * ncol, 4 * len(ids)))
axes = np.atleast_2d(axes)
for r, i in enumerate(ids):
    cols = [(DATA_DIR / "lensless" / f"{i}.png", "lensless measurement"),
            (recon_dir / f"{i}.png", "reconstruction (ours)")]
    if has_gt:
        cols.append((DATA_DIR / "lensed" / f"{i}.png", "ground truth (lensed)"))
    for c, (path, title) in enumerate(cols):
        axes[r, c].imshow(Image.open(path))
        axes[r, c].set_title(title)
        axes[r, c].axis('off')
plt.tight_layout()
plt.show()

## 6. Quantitative metrics (only if ground truth is present)

In [ ]:
if has_gt:
    os.system(f"python calculate_metrics.py --gt {DATA_DIR}/lensed --pred {recon_dir}")
else:
    print("No ground truth in this dataset")

## Bonus models 

Run a bonus reconstruction on the same data — set `BONUS` to `"fista"`, `"esrgan_frozen"`, or `"esrgan_ft"`:
- **fista** — classical FISTA solver, no weights
- **esrgan_frozen** — ADMM-100 + pretrained Real-ESRGAN 
- **esrgan_ft** — ADMM-100 + our fine-tuned Real-ESRGAN 

In [ ]:
BONUS = "fista" # choose "fista"/"esrgan_frozen"/"esrgan_ft"
GAN_URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

BONUS_CFG = {
    "fista": {"model": "fista", "ckpt": None, "gan_weights": False, "extra": ""},
    "esrgan_frozen": {"model": "admm_realesrgan", "ckpt": None, "gan_weights": True, "extra": ""},
    "esrgan_ft": {"model": "admm_realesrgan_ft", "ckpt": "admm-esrgan-ft.pth", "gan_weights": False, "extra": "model.post.weights_path=null"},
}
cfg = BONUS_CFG[BONUS]

if cfg["gan_weights"] and not Path("weights/RealESRGAN_x4plus.pth").exists():
    Path("weights").mkdir(exist_ok=True)
    os.system(f"curl -L -o weights/RealESRGAN_x4plus.pth {GAN_URL}")

ckpt = hf_hub_download(repo_id=CKPT_REPO, filename=cfg["ckpt"]) if cfg["ckpt"] else None

RECON_B = str(REPO_DIR / "reconstructions" / BONUS)
load = f"inferencer.skip_model_load=false inferencer.from_pretrained={ckpt}" if ckpt else "inferencer.skip_model_load=true"
cmd = (
    f"python inference.py --config-name=inference_admm100 model={cfg['model']} {cfg['extra']} "
    f"datasets=custom_dir datasets.test.data_dir={DATA_DIR} dataloader.batch_size=1 "
    f"{load} inferencer.save_path={RECON_B}"
)
os.system(cmd)
print(f"{BONUS} reconstructions at: ", Path(RECON_B, "test").resolve())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

recon_dir_b = Path(RECON_B, "test")
has_gt = (DATA_DIR / "lensed").exists()
ids = sorted(p.stem for p in recon_dir_b.glob("*.png"))[:4]
ncol = 3 if has_gt else 2
fig, axes = plt.subplots(len(ids), ncol, figsize=(4 * ncol, 4 * len(ids)))
axes = np.atleast_2d(axes)
for r, i in enumerate(ids):
    cols = [(DATA_DIR / "lensless" / f"{i}.png", "lensless measurement"),
            (recon_dir_b / f"{i}.png", f"{BONUS} reconstruction")]
    if has_gt:
        cols.append((DATA_DIR / "lensed" / f"{i}.png", "ground truth (lensed)"))
    for c, (path, title) in enumerate(cols):
        axes[r, c].imshow(Image.open(path))
        axes[r, c].set_title(title)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

if has_gt:
    os.system(f"python calculate_metrics.py --gt {DATA_DIR}/lensed --pred {recon_dir_b}")